# 00 · El proceso completo

**Duración estimada:** 90 minutos

## Propósito

El práctico está partido en cuatro cuadernos porque una clase dura lo que dura. Pero el
proceso no tiene cuatro partes: es uno solo, y va del archivo crudo a la interpretación.

Acá lo vas a recorrer **entero, de arriba a abajo, en un solo cuaderno**. Y después vas a
agregarle las tres operaciones que la unidad no llegó a hacer:

1. **La concordancia** — volver del número al texto.
2. **El análisis de sentimiento** — poner un signo a lo que dice el corpus.
3. **El modelado de tópicos** — dejar que los temas salgan solos, y ponerles nombre vos.

Se hace **al final**, cuando ya pasaste por los cuatro cuadernos del práctico. Antes no
tiene gracia: lo que este cuaderno muestra es que esas cuatro partes eran una sola cosa.

## Qué consume

- `../practico/datos/blog/` — los doce archivos originales
- `../practico/datos/stopwords_es.txt` — la lista de palabras vacías
- `datos/lexicon_afectivo_es.csv` — el lexicón afectivo de la cátedra
- `utils.py` — el archivo de al lado

## Qué produce

- `analisis_del_corpus.py` — un script que corre el pipeline sin Jupyter

## Una advertencia sobre este cuaderno

Corre de principio a fin con **Restart Kernel and Run All Cells**, y tarda alrededor de un
minuto en las celdas del modelado. No hace falta ejecutar nada fuera de orden.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Pipeline** | La cadena de etapas por las que pasa el texto hasta volverse un resultado |
| **Concordancia** | Todas las apariciones de algo en un corpus, con su contexto alrededor |
| **KWIC** | *Key word in context*. Otro nombre para la concordancia |
| **Nodo** | El término que buscás. En la tabla va entre corchetes |
| **Cotexto** | El texto que rodea al nodo. Distinto de *contexto*, que es la situación |
| **Ventana** | Cuántos caracteres del cotexto se muestran a cada lado |

> **Ojo con la palabra.** *Concordancia* también significa otra cosa en esta carrera: el
> acuerdo entre dos personas que anotan el mismo material. En la unidad de spaCy vas a
> encontrarla con ese sentido. Acá no es eso.

## Cómo está armado

| Parte | Qué hace | Dirección |
|---|---|---|
| 1 a 9 | El proceso del práctico, completo y en orden | del texto al número |
| 10 | La concordancia | del número al texto |
| 11 | El sentimiento | del texto a un signo |
| 12 | Los tópicos | del texto a los temas |
| 13 | `utils.py` y el script | del cuaderno al programa |

Las nueve primeras secciones **van del texto al número**: cortan, limpian, cuentan,
vectorizan, comparan, dibujan. Cada una recibe lo que produjo la anterior.

La décima va en la otra dirección, y es la que cambia todo lo demás.

### ⚙ Configuración del entorno

Una sola celda de importaciones para todo el cuaderno. Si alguna falla, revisá
`ANTES_DE_LA_CLASE.md`: falta el entorno virtual.

In [ ]:
# pathlib: para armar rutas que funcionen en cualquier sistema operativo.
import pathlib

# pickle: para leer objetos de Python guardados en disco.
import pickle

# re: expresiones regulares, para buscar patrones en el texto.
import re

# Counter: para contar cosas. Devuelve un diccionario cosa -> veces.
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# La semilla fija el azar de todo lo que lo use. Sin esto, el cuaderno no es reproducible.
SEMILLA = 42
np.random.seed(SEMILLA)

print(f"pandas {pd.__version__} — numpy {np.__version__}")
print("Semilla fijada. El cuaderno es reproducible.")

El corpus vive en el práctico y **no se duplica**. Este cuaderno lo lee de ahí.

In [ ]:
# La carpeta del práctico, vista desde acá.
CARPETA_DATOS = pathlib.Path("..") / "practico" / "datos"

# Los datos propios del anexo: por ahora, el lexicón afectivo.
CARPETA_ANEXO = pathlib.Path("datos")

# Los doce años del corpus.
anios = list(range(2004, 2016))

print("corpus en   :", CARPETA_DATOS / "blog")
print("¿existe?    :", (CARPETA_DATOS / "blog").exists())
print("años a leer :", anios[0], "a", anios[-1])

## 1. Configurar los gráficos una sola vez

Los valores por omisión de matplotlib no son buenos: rejillas gruesas, marcos por los
cuatro lados, letra chica. Es la misma configuración del cuaderno `03` del práctico, así
que los gráficos de acá van a salir iguales a los de allá.

In [ ]:
# --- Los colores de la unidad ---

# La superficie: un blanco apenas cálido, menos duro que el blanco puro.
SUPERFICIE = "#fcfcfb"

# La tinta: un gris muy oscuro con una gota de violeta, en vez de negro puro.
TINTA = "#3d3a4a"

# Dos colores para los datos: uno principal y uno de acento.
VIOLETA = "#6437b5"
ROSA = "#d1568c"

# Y un neutro, para lo que queremos que se lea pero no se destaque.
NEUTRO = "#8d8a99"

# Una rampa ordenada, de lo más frecuente a lo menos.
RAMPA = ["#2b1b6b", "#5b2ea6", "#8f3bbf", "#c4419f", "#e0557b"]

print("paleta lista")

In [ ]:
# --- El fondo y el marco ---

# Tamaño por omisión de las figuras, en pulgadas.
plt.rcParams["figure.figsize"] = (8, 4)

# El fondo, en los dos lugares donde matplotlib lo pinta.
plt.rcParams["figure.facecolor"] = SUPERFICIE
plt.rcParams["axes.facecolor"] = SUPERFICIE

# Sacamos el marco de arriba y de la derecha: no aportan información.
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.edgecolor"] = "#c9c6d0"

print("fondo y marco listos")

In [ ]:
# --- La rejilla y el texto ---

# Rejilla fina y clarita, siempre por detrás de los datos.
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.color"] = "#d8d5de"
plt.rcParams["grid.linewidth"] = 0.7
plt.rcParams["axes.axisbelow"] = True

# Todo el texto en la misma tinta.
plt.rcParams["font.size"] = 10
plt.rcParams["text.color"] = TINTA
plt.rcParams["axes.labelcolor"] = TINTA
plt.rcParams["axes.titlecolor"] = TINTA
plt.rcParams["xtick.color"] = TINTA
plt.rcParams["ytick.color"] = TINTA

print("configuración de gráficos lista")

## 2. Etapa uno: del archivo al corpus

**La analogía.** Un corpus es un archivo de expedientes. Lo que nos entregaron no es eso:
es una pila de doce sobres cerrados, sin índice, sin fechas y con la etiqueta equivocada.

**Dónde vive esto.** Cualquier trabajo con textos reales empieza así. Los datos de
lenguaje casi nunca llegan listos: llegan de un scrapeo que alguien hizo apurado, de una
exportación de un sistema viejo, de un volcado de base de datos. La primera tarea nunca es
analizar. Es **averiguar qué te dieron**.

Empecemos mirando los archivos.

In [ ]:
carpeta = CARPETA_DATOS / "blog"
archivos = sorted(carpeta.glob("*.txt"))

print("archivos encontrados:", len(archivos))
print()
for archivo in archivos:
    tamano = archivo.stat().st_size
    print(archivo.name, "->", f"{tamano:,}", "bytes")

Doce archivos con extensión `.txt`. En el cuaderno `00` del práctico descubrimos que **la
extensión miente**: no son texto, son objetos `pickle`. Los primeros bytes de cada uno son
la firma `\x80\x03X` del formato.

Por eso la función que los lee usa el modo binario `"rb"` y `pickle.load`, y no `.read()`.

> **Advertencia:** abrir un pickle **ejecuta código**. Estos archivos vienen con el
> material de la materia; uno que te llegue de un desconocido no se abre.

In [ ]:
def cargar_anio(anio):
    """Lee uno de los archivos de blog/. No son texto: son objetos pickle."""
    ruta = CARPETA_DATOS / "blog" / (str(anio) + ".txt")
    archivo = open(ruta, "rb")
    texto = pickle.load(archivo)
    archivo.close()
    return texto


texto_2004 = cargar_anio(2004)

cantidad_de_caracteres = len(texto_2004)
print("tipo      :", type(texto_2004).__name__)
print("caracteres:", f"{cantidad_de_caracteres:,}")
print()
print(texto_2004[:200])

Un año entero es **un solo string gigante**. No hay párrafos, ni títulos, ni fechas: el
scrapeo original bajó los textos por separado y los pegó, y esa información se perdió.

Pero dejó una huella. Mirá este tramo.

In [ ]:
print(texto_2004[380:520])

Fijate en `lector.La idea nació`: **un punto pegado a una mayúscula, sin espacio en el
medio.** Eso no pasa cuando alguien escribe. Pasa cuando se pegan dos párrafos y se borra
el salto de línea.

Una de cada cuatro fronteras de oración del corpus está así. Cada una marca dónde
terminaba un párrafo, y eso permite reconstruir una unidad intermedia entre *el año
entero* y *el texto individual que se perdió*.

In [ ]:
def cortar_en_parrafos(texto):
    """Corta el bloque de un año en párrafos, donde un punto pega con una mayúscula."""
    # El punto NO se consume: por eso cada párrafo conserva su puntuación final.
    marca = r'(?<=[a-záéíóúñ0-9\)\!\?\»"])\.(?=[A-ZÁÉÍÓÚÑ¿«"])'
    partes = re.split(marca, texto)

    parrafos = []
    for parte in partes:
        limpio = parte.strip()
        if len(limpio) < 2:
            continue
        if not limpio.endswith((".", "!", "?", "…", "»", '"')):
            limpio = limpio + "."
        parrafos.append(limpio)
    return parrafos


parrafos_2004 = cortar_en_parrafos(texto_2004)

print("párrafos en 2004:", len(parrafos_2004))
print()
print(parrafos_2004[3])

### ✎ Antes de ejecutar

La celda siguiente arma la tabla con los doce años. ¿Cuántas filas esperás que tenga?
El corpus son 2,4 millones de caracteres. Escribí tu estimación antes de correrla.

In [ ]:
def armar_corpus(anios):
    """Devuelve una tabla con un párrafo por fila, y de qué año es cada uno."""
    filas = []
    for anio in anios:
        texto_del_anio = cargar_anio(anio)
        for parrafo in cortar_en_parrafos(texto_del_anio):
            filas.append({"anio": anio, "texto": parrafo})
    return pd.DataFrame(filas)


corpus = armar_corpus(anios)

cantidad_de_parrafos = len(corpus)
# '.tolist()' saca los números de numpy y los deja como enteros de Python.
anios_del_corpus = sorted(corpus["anio"].unique().tolist())

print("párrafos:", f"{cantidad_de_parrafos:,}")
print("años    :", anios_del_corpus)
corpus.head(3)

**5.428 párrafos.** Es el mismo número que da el cuaderno `00` del práctico, y era de
esperar: es el mismo corte sobre los mismos archivos.

Antes de seguir, dejemos una aserción que verifique el estado. Si mañana alguien cambia
`cortar_en_parrafos` y el corpus se rompe, esta celda va a avisar.

In [ ]:
assert isinstance(corpus, pd.DataFrame), "El corpus tiene que ser un DataFrame."
assert len(corpus) == 5428, "El corte en párrafos cambió: ya no dan 5.428 filas."
assert list(corpus.columns) == ["anio", "texto"], "Las columnas esperadas son anio y texto."
assert corpus["anio"].nunique() == 12, "Faltan años en el corpus."

print("Etapa 1 verificada: 5.428 párrafos, 12 años, dos columnas.")

Eso es la **etapa 1**. En el práctico ocupa el cuaderno `00` entero; acá son tres
funciones. El trabajo no desapareció: está en las funciones.

### ✎ Para pensar

- Con este corpus **no se puede** preguntar en cuántos textos distintos aparece una
  palabra. ¿Por qué?
- El párrafo es una unidad que **nosotros inventamos** con una expresión regular. ¿Qué
  pasaría con el análisis si el corte fuera por oración en vez de por párrafo?

## 3. Etapa dos: del corpus al texto limpio

**La analogía.** Limpiar un texto es como pasar una fotocopia a blanco y negro puro: se
gana legibilidad para cierta tarea y se pierde todo lo que estaba en los grises.

Para contar palabras, `Casa`, `casa` y `casa,` tienen que ser la misma cosa. Para leer una
cita, no.

In [ ]:
# Los signos que saca limpiar(). Incluye los de apertura del español (¡ ¿) y las
# comillas y guiones tipográficos, que string.punctuation no trae.
SIGNOS = "¡!\"#$%&'()*+,-./:;<=>¿?@[\\]^_`{|}~«»…—–“”‘’"
PATRON_DE_SIGNOS = "[" + re.escape(SIGNOS) + "]"

print("signos que se van:", SIGNOS)
print("cantidad         :", len(SIGNOS))

Con el patrón armado, la función de limpieza son tres líneas.

In [ ]:
def limpiar(texto):
    """Pasa a minúsculas, saca la puntuación y unifica los espacios."""
    en_minusculas = texto.lower()
    sin_signos = re.sub(PATRON_DE_SIGNOS, " ", en_minusculas)
    sin_espacios_de_mas = re.sub(r"\s+", " ", sin_signos)
    return sin_espacios_de_mas.strip()


ejemplo = corpus["texto"].iloc[3]

print("ORIGINAL:")
print(ejemplo[:160])
print()
print("LIMPIO:")
print(limpiar(ejemplo)[:160])

### La operación que no vamos a hacer

El material del que salió este ejercicio tenía una cuarta línea de limpieza:

```python
texto = re.sub(r'\w*\d\w*', ' ', texto)   # borrar las palabras con números
```

Parece razonable: saca códigos y basura numérica. Midamos qué hace **en este corpus**.

In [ ]:
todo_el_corpus = " ".join(corpus["texto"])

# Años de cuatro dígitos: 1999, 2001, 2014...
menciones_a_anios = re.findall(r"\b(?:19|20)\d{2}\b", todo_el_corpus)

cantidad_de_menciones = len(menciones_a_anios)
print("menciones a años en el corpus:", f"{cantidad_de_menciones:,}")
print()
conteo_de_menciones = Counter(menciones_a_anios)
for anio_mencionado, veces in conteo_de_menciones.most_common(6):
    print(" ", anio_mencionado, "->", veces)

**Esa línea borraría 447 menciones a años** en un corpus cuya única variable es el tiempo.
*"En 2001 gané la F1"* queda *"en gané la"*.

No la usamos. Y el punto no es que esté mal en general —en otro corpus puede ser justo lo
que hace falta— sino que **toda operación de limpieza destruye algo, y hay que saber qué.**

> **Limpiar no es corregir errores.** Es decidir qué diferencias importan y cuáles no.
> Cada línea de limpieza es una afirmación sobre el corpus.

In [ ]:
textos_limpios = []
for texto in corpus["texto"]:
    textos_limpios.append(limpiar(texto))

corpus["texto_limpio"] = textos_limpios

assert "texto" in corpus.columns, "El texto original no puede desaparecer."
assert "texto_limpio" in corpus.columns, "Falta la columna con el texto limpio."

print("columnas del corpus:", list(corpus.columns))
corpus[["anio", "texto", "texto_limpio"]].head(2)

Fijate que el corpus guarda **las dos columnas**.

Eso no es desprolijidad. La limpieza destruye la puntuación y las mayúsculas a propósito,
porque estorban para contar. Pero son justo lo que hace legible una cita. Si tirás el
original, no hay manera de volver.

**Guardá siempre el texto sin tocar.** En la sección 10 vas a ver por qué esto es lo más
importante del cuaderno.

## 4. Etapa tres: del texto limpio a la cuenta

Ya podemos contar. Pero antes hacen falta las **palabras vacías**: la lista de términos
que se descartan porque son frecuentes en todo el idioma y no hablan de ningún tema.

In [ ]:
def cargar_palabras_vacias():
    """La lista de palabras vacías. En el práctico se carga una vez por cuaderno."""
    ruta = CARPETA_DATOS / "stopwords_es.txt"
    archivo = open(ruta, encoding="utf-8")
    contenido = archivo.read()
    archivo.close()
    return contenido.split()


palabras_vacias = cargar_palabras_vacias()

print("palabras vacías en la lista:", len(palabras_vacias))
print()
print("¿está 'vos'?", "vos" in palabras_vacias)
print("¿está 'tú'? ", "tú" in palabras_vacias)

`tú` está y `vos` no. La lista se armó pensando en el español peninsular, y en un corpus
rioplatense eso significa que `vos` va a quedar entre las palabras más frecuentes de tu
análisis aunque sea un pronombre igual que `tú`.

No es un error de la lista: **una lista de palabras vacías siempre viene de algún lado y
trae los supuestos de ese lugar.**

In [ ]:
def agrupar_por_anio(corpus):
    """Junta las palabras limpias de cada año. Devuelve un diccionario por año."""
    palabras_por_anio = {}
    for anio in sorted(corpus["anio"].unique()):
        parrafos_del_anio = corpus[corpus["anio"] == anio]
        texto_del_anio = " ".join(parrafos_del_anio["texto_limpio"])
        palabras_por_anio[anio] = texto_del_anio.split()
    return palabras_por_anio


palabras_por_anio = agrupar_por_anio(corpus)

for anio in anios:
    cantidad = len(palabras_por_anio[anio])
    print(anio, "->", f"{cantidad:,}", "palabras")

Los años **no son comparables en tamaño**: 2008 tiene 55.588 palabras y 2009 tiene 14.838.
Casi cuatro veces menos.

Esa diferencia es la trampa más grande de la unidad, y por eso la función que cuenta
devuelve **dos columnas y no una**.

In [ ]:
def frecuencias_de(palabras_por_anio, palabra_buscada):
    """Para cada año: cuántas veces aparece la palabra, cruda y por cada 10.000."""
    filas = []
    for anio in sorted(palabras_por_anio):
        palabras = palabras_por_anio[anio]
        veces = palabras.count(palabra_buscada)
        por_diez_mil = 10000 * veces / len(palabras)
        filas.append({"anio": anio, "veces": veces, "por_10k": round(por_diez_mil, 2)})
    return pd.DataFrame(filas)


tabla_padre = frecuencias_de(palabras_por_anio, "padre")
tabla_padre

Ahí está la lección del cuaderno `01` en dos columnas: **`veces` compara tamaños de
archivo, `por_10k` compara textos.**

2012 tiene 2 apariciones y 2004 tiene 25: parece una diferencia de doce veces. Pero por
cada diez mil palabras, 2012 tiene 1,01 y 2004 tiene 7,32. La diferencia real es de
**siete veces**, no de doce.

Dibujemos las dos series juntas para ver cuánto cambia la forma.

In [ ]:
figura, paneles = plt.subplots(1, 2, figsize=(11, 3.6))

paneles[0].bar(tabla_padre["anio"], tabla_padre["veces"], color=NEUTRO)
paneles[0].set_title("Conteo crudo: 'padre'")
paneles[0].set_ylabel("veces")

paneles[1].bar(tabla_padre["anio"], tabla_padre["por_10k"], color=VIOLETA)
paneles[1].set_title("Normalizado: 'padre' por cada 10.000 palabras")
paneles[1].set_ylabel("por 10k")

plt.tight_layout()
plt.show()

Los mismos datos. **El año más alto no es el mismo en los dos gráficos.**

En el crudo gana 2005; en el normalizado gana 2015. El primero mide en qué año hay más
texto guardado. El segundo mide en qué año Casciari habló más de padres.

### ✎ Para pensar

- Si tuvieras que titular el gráfico de la izquierda sin mentir, ¿qué diría el título?
- ¿En qué caso el conteo crudo **sí** es la medida correcta?

## 5. La matriz documento-término

Contar una palabra por vez sirve para una pregunta. Para comparar todo contra todo hace
falta la estructura del teórico: la **matriz documento-término**, una fila por documento y
una columna por palabra del vocabulario.

Y acá aparece la decisión más importante de esta parte: **qué es un documento.** Doce años
o cinco mil párrafos. Vamos con los años, que es lo natural si la pregunta es sobre el
paso del tiempo.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

documentos_por_anio = []
for anio in anios:
    documentos_por_anio.append(" ".join(palabras_por_anio[anio]))

# 'stop_words' descarta la lista de palabras vacías.
# 'min_df=2' descarta lo que aparece en un solo documento: baja el ruido y también
# borra lo muy específico de un año. Es una decisión, no un valor neutro.
contador = CountVectorizer(stop_words=palabras_vacias, min_df=2)
matriz_por_anio = contador.fit_transform(documentos_por_anio)

print("forma de la matriz:", matriz_por_anio.shape)
print("  12 documentos por", f"{matriz_por_anio.shape[1]:,}", "palabras distintas")

### ✎ Antes de ejecutar

Doce documentos por trece mil palabras son más de 160.000 casilleros. ¿Qué porcentaje te
parece que tiene un número distinto de cero?

In [ ]:
celdas_totales = matriz_por_anio.shape[0] * matriz_por_anio.shape[1]

# 'nnz' es "number of non-zero": cuántos casilleros tienen algo.
celdas_con_valor = matriz_por_anio.nnz
porcentaje_lleno = 100 * celdas_con_valor / celdas_totales

print("casilleros en total :", f"{celdas_totales:,}")
print("casilleros con valor:", f"{celdas_con_valor:,}")
print("porcentaje lleno    :", f"{porcentaje_lleno:.1f}%")

Un 36% lleno: dos de cada tres casilleros están en cero. Por eso `CountVectorizer` no
devuelve una tabla común sino una **matriz esparsa**, que guarda solamente los valores que
no son cero.

Y ese 36% es *alto* porque son doce documentos enormes. Con los párrafos como unidad, la
misma matriz queda casi vacía —lo vas a ver en la sección 12, cuando el modelado de
tópicos necesite justamente esa otra unidad.

In [ ]:
# La convertimos a tabla solo para mirarla: con 12 filas se puede.
vocabulario_por_anio = contador.get_feature_names_out()
tabla_frecuencias = pd.DataFrame(
    matriz_por_anio.toarray(),
    columns=vocabulario_por_anio,
    index=anios,
)

memoria_en_mb = tabla_frecuencias.memory_usage(deep=True).sum() / 1024 ** 2
print("forma :", tabla_frecuencias.shape)
print("memoria:", f"{memoria_en_mb:.1f}", "MB")

# Cinco columnas elegidas a mano. Si tomáramos cinco cualquiera del medio del
# vocabulario saldrían números: 'limpiar' no borra los dígitos, y el vocabulario
# arranca con 33, 34, 35, 350, 36.
tabla_frecuencias[["casa", "revista", "orsai", "padre", "lucas"]]

Cada número es *cuántas veces esa palabra aparece en ese año*, y ya se lee algo: `revista`
y `orsai` se disparan a partir de 2010, `lucas` está en 2004 y después casi desaparece.

Y fijate en el comentario de la celda: **el vocabulario empieza con `33`, `34`, `35`,
`350`, `36`.** Son los números y los años que `limpiar` no borró, a propósito (sección 3).
Para contar palabras no molestan; para leer una lista alfabética de vocabulario, sí.

**Ojo con `.toarray()`.** Convertir la matriz esparsa a tabla común funciona con doce
documentos. Con cinco mil párrafos y treinta mil palabras, la misma línea pide más de un
gigabyte de memoria y la máquina se cae.

## 6. Las palabras vacías del corpus

La lista de palabras vacías es general del español. Pero cada corpus tiene **sus propias**
palabras que aparecen en todas partes y no distinguen nada.

El material original las buscaba así: tomar las 30 más frecuentes de cada año y quedarse
con las que aparecen en más de la mitad de los años.

In [ ]:
def contar_con_contenido(palabras, palabras_vacias, largo_minimo=3):
    """Cuenta las palabras de un año, salteando las vacías y las muy cortas."""
    conteo = Counter()
    for palabra in palabras:
        if palabra in palabras_vacias:
            continue
        if len(palabra) < largo_minimo:
            continue
        conteo[palabra] = conteo[palabra] + 1
    return conteo


conteos_por_anio = {}
totales_con_contenido = {}
for anio in anios:
    conteo_del_anio = contar_con_contenido(palabras_por_anio[anio], palabras_vacias)
    conteos_por_anio[anio] = conteo_del_anio
    totales_con_contenido[anio] = sum(conteo_del_anio.values())

print("palabras con contenido en 2004:", f"{totales_con_contenido[2004]:,}")

Ahora contamos en cuántos de los doce tops aparece cada palabra.

In [ ]:
apariciones_en_los_tops = Counter()
for anio in anios:
    for palabra, veces in conteos_por_anio[anio].most_common(30):
        apariciones_en_los_tops[palabra] = apariciones_en_los_tops[palabra] + 1

candidatas = []
for palabra, en_cuantos_anios in apariciones_en_los_tops.most_common():
    if en_cuantos_anios > 6:
        candidatas.append({"palabra": palabra, "en_cuantos_anios": en_cuantos_anios})

tabla_candidatas = pd.DataFrame(candidatas)
print("candidatas a palabra vacía del corpus:", len(tabla_candidatas))
tabla_candidatas

Veintidós candidatas. El material original agregaba esta lista a las palabras vacías y
volvía a vectorizar.

**Nosotros vamos a mirarlas primero.** Están `vez`, `cada`, `dos`, `ahora`, `después`: son
ruido, sin discusión. Pero también están `casa`, `vida`, `mundo`, `tiempo`... y `chiri`.

`chiri` no es una palabra vacía: es **una persona**. Es el apodo de Christian Basilis, el
amigo con el que Casciari escribe desde el secundario y con el que después funda la
revista Orsai. Comprobémoslo.

In [ ]:
frecuencias_de(palabras_por_anio, "chiri")

Aparece los doce años, siempre. Por eso el criterio automático la marcó: **el criterio
busca lo que está en todas partes, y un personaje central está en todas partes.**

Si la agregás a la lista de palabras vacías, el análisis pierde al coautor del corpus.

> **La regla que queda.** Una lista de palabras vacías construida automáticamente es una
> **propuesta**, no un resultado. Se lee entera antes de aplicarla, y lo que se descarta
> se anota.

Decisión de este cuaderno: **seguimos con la lista base**. Las candidatas quedan
documentadas y en la sección 12 vas a poder medir qué cambia si las usás.

### ✎ Para pensar

- `casa`, `vida` y `mundo` aparecen en el top de diez años o más. ¿Son ruido o son tema?
- Si el corpus fuera de expedientes judiciales, ¿qué palabra ocuparía el lugar de `chiri`?

## 7. Etapa cuatro: comparar un año contra el resto

La pregunta del práctico `02`: **¿qué palabras caracterizan a cada año?**

Ahí medimos que TF-IDF casi no sirve para esto: con doce documentos el IDF solo puede
tomar once valores distintos, y un tercio del vocabulario comparte el mismo. La
herramienta que sí sirve se llama **keyness** y no necesita ninguna librería.

La idea cabe en una frase: *una palabra caracteriza a un año si aparece mucho más en ese
año que en todos los demás juntos.*

In [ ]:
def caracteristicas_de(conteos_por_anio, totales, anio_elegido,
                       minimo_de_apariciones=10, cuantas=10):
    """Qué palabras distinguen a un año del resto del corpus."""
    conteo_del_resto = Counter()
    total_del_resto = 0
    for anio in conteos_por_anio:
        if anio == anio_elegido:
            continue
        conteo_del_resto.update(conteos_por_anio[anio])
        total_del_resto = total_del_resto + totales[anio]

    filas = []
    for palabra in conteos_por_anio[anio_elegido]:
        veces = conteos_por_anio[anio_elegido][palabra]
        if veces < minimo_de_apariciones:
            continue
        aca = 10000 * veces / totales[anio_elegido]
        alla = 10000 * conteo_del_resto[palabra] / total_del_resto
        # El 0,5 evita dividir por cero cuando la palabra no está en el resto.
        filas.append({"palabra": palabra, "veces": veces,
                      "por_10k_este_anio": round(aca, 1),
                      "por_10k_el_resto": round(alla, 1),
                      "cuantas_veces_mas": round(aca / (alla + 0.5), 1)})

    tabla = pd.DataFrame(filas)
    tabla = tabla.sort_values(by="cuantas_veces_mas", ascending=False)
    return tabla.head(cuantas).reset_index(drop=True)

Probémosla con 2010, el año en que arrancó la revista.

In [ ]:
caracteristicas_de(conteos_por_anio, totales_con_contenido, 2010, cuantas=8)

*Hornby*, *libreros*, *librerías*, *revista*, *picnic*.

2010 es el año en que arrancó la revista Orsai, y Nick Hornby fue uno de los autores
publicados. La tabla no sabe nada de eso: lo dedujiste vos, porque conocés algo que el
corpus no dice.

**Ese salto —de la tabla a la explicación— es el único lugar donde hace falta una
persona.** Veamos los doce años juntos.

In [ ]:
filas = []
for anio in anios:
    tabla_del_anio = caracteristicas_de(conteos_por_anio, totales_con_contenido,
                                        anio, cuantas=6)
    palabras_del_anio = ", ".join(tabla_del_anio["palabra"])
    filas.append({"anio": anio, "palabras_caracteristicas": palabras_del_anio})

pd.DataFrame(filas)

Lo primero que salta es que **casi todo son nombres propios**: `mónica`, `bernardo`,
`karen`, `alfonso`, `eugenia`, `paola`, `moncho`. Tiene sentido, y no es un defecto: un
personaje que aparece en los relatos de un año y en ninguno más es, literalmente, lo más
característico que ese año puede tener.

Entre los nombres se filtra la otra historia.

| Año | Palabra que no es un nombre |
|---|---|
| 2010 | `libreros`, `librero`, `librerías` |
| 2011 | `número` |
| 2012 | `suscripción` |
| 2014 | `taller`, `talleres`, `participantes` |

Son los años en que Casciari funda Editorial Orsai y se pone a publicar. **La misma
historia va a volver dos veces más en este cuaderno**, en la sección 11 y en la 12, por
caminos completamente distintos. Guardate esta tabla.

> ### Perilla
>
> `caracteristicas_de` tiene dos parámetros y los dos cambian el resultado.
>
> **`minimo_de_apariciones`** filtra las palabras raras. Con `2`, una palabra que aparece
> dos veces en un año y nunca más encabeza el ranking, y no caracteriza nada: es una
> anécdota. Con `50` solo sobreviven las muy frecuentes y desaparece lo específico.
>
> **`cuantas`** es cuántas mostrás. Ahí no hay trampa técnica, pero sí una tentación:
> mostrar solo las que confirman lo que querías decir.
>
> Corré la celda y mirá cómo cambia la historia de 2013 con cada mínimo.

In [ ]:
for minimo in [3, 10, 30]:
    tabla_del_anio = caracteristicas_de(conteos_por_anio, totales_con_contenido, 2013,
                                        minimo_de_apariciones=minimo, cuantas=6)
    print("mínimo =", minimo, "->", ", ".join(tabla_del_anio["palabra"]))

## 8. Las estadísticas del corpus, y un número que no se puede usar

El material original cerraba con una tabla de estadísticas por año: posts, palabras
únicas, promedio de palabras por post, diversidad léxica. Empezaba con esta línea:

```python
posts_por_anio = [50, 27, 18, 50, 42, 22, 50, 33, 31, 17, 33, 13]
```

**Ese número no está en el corpus.** Venía del análisis original del blog, cuando los
textos todavía estaban separados. Nosotros medimos en la sección 2 que esa separación se
perdió: acá no hay manera de contar los posts de un año.

Un dato cableado que no se puede verificar contra la fuente es exactamente lo que la
unidad entera enseña a no aceptar. Lo reemplazamos por **párrafos**, que sí podemos contar
porque los construimos nosotros.

In [ ]:
filas = []
for anio in anios:
    parrafos_del_anio = corpus[corpus["anio"] == anio]
    palabras = palabras_por_anio[anio]
    palabras_unicas = len(set(palabras))
    filas.append({
        "anio": anio,
        "parrafos": len(parrafos_del_anio),
        "palabras": len(palabras),
        "vocabulario": palabras_unicas,
        "palabras_por_parrafo": round(len(palabras) / len(parrafos_del_anio), 1),
        "diversidad_lexica": round(palabras_unicas / len(palabras), 4),
    })

estadisticas = pd.DataFrame(filas)
estadisticas

La **diversidad léxica** es la razón entre palabras distintas y palabras totales. Se lee
como *riqueza de vocabulario*: cuanto más alta, más variado el léxico.

Mirá la columna: el año más "rico" es **2009, con 0,2486**. Y el más "pobre" es **2005,
con 0,1642**.

### ✎ Antes de ejecutar

Antes de aceptar esa lectura: 2009 tiene 14.838 palabras y 2005 tiene 56.058. ¿Se te
ocurre por qué el año más chico podría salir siempre más "rico"?

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(estadisticas["palabras"], estadisticas["diversidad_lexica"],
            color=VIOLETA, s=60, zorder=3)

for fila in estadisticas.itertuples():
    plt.annotate(str(fila.anio), xy=(fila.palabras, fila.diversidad_lexica),
                 xytext=(5, 4), textcoords="offset points", fontsize=8, color=NEUTRO)

plt.xlabel("palabras del año")
plt.ylabel("diversidad léxica")
plt.title("La diversidad léxica baja cuando el texto crece")
plt.tight_layout()
plt.show()

**Los doce puntos caen en una curva.** No hay ni un año grande con diversidad alta.

La razón es aritmética y no tiene nada que ver con Casciari: las palabras nuevas se agotan.
Los primeros mil tokens de cualquier texto traen cientos de palabras nunca vistas; los
últimos mil traen unas pocas. Entonces **cuanto más largo el texto, más baja la razón**,
siempre.

Es el mismo error que el conteo crudo de la sección 4, con otro disfraz: una medida que
depende del tamaño usada para comparar cosas de tamaños distintos.

> **Cómo se resuelve.** Existen medidas pensadas para esto —la más conocida se llama MTLD—
> y el arreglo casero es recortar todos los años al mismo largo antes de comparar. Lo que
> **no** se puede hacer es lo que hacía la tabla original: leer esa columna como si dijera
> algo sobre el estilo del autor.

### ✎ Para pensar

- ¿Qué afirmación **sí** sostiene la columna `palabras_por_parrafo`?
- 2009 tiene 116 párrafos contra 813 de 2008. ¿Eso es un dato sobre Casciari o sobre el
  scrapeo?

## 9. Mostrar los resultados

**La analogía.** Una nube de palabras es la portada de una revista: te dice de qué se
trata el número, y no te permite verificar nada.

Es lo que todo el mundo espera de un análisis de texto, así que hagámosla —y después las
preguntas incómodas.

In [ ]:
# wordcloud: la librería de nubes de palabras.
from wordcloud import WordCloud
from matplotlib import font_manager

# Buscamos una tipografía liviana; si no está instalada, seguimos con la que haya.
RUTA_FUENTE = None
for nombre_de_fuente in ["Carlito", "Liberation Sans", "DejaVu Sans"]:
    try:
        RUTA_FUENTE = font_manager.findfont(nombre_de_fuente, fallback_to_default=False)
        break
    except Exception:
        continue

print("tipografía:", RUTA_FUENTE)

Ya tenemos tipografía. La nube del corpus entero, con la paleta de la unidad.

In [ ]:
nube = WordCloud(
    width=900, height=420,
    background_color=SUPERFICIE,
    colormap="Purples",
    stopwords=set(palabras_vacias),
    max_words=60,
    random_state=SEMILLA,
    font_path=RUTA_FUENTE,
)
nube.generate(" ".join(corpus["texto_limpio"]))

plt.figure(figsize=(10, 4.6))
plt.imshow(nube, interpolation="bilinear")
plt.axis("off")
plt.grid(False)
plt.show()

Es linda. Ahora las preguntas del cuaderno `03`.

**¿Qué afirmación sostiene esta imagen?** Ninguna en particular. Muestra que ciertas
palabras son frecuentes en un corpus de doce años. No dice cuánto, ni comparado con qué,
ni si eso cambió.

**¿Qué número podés verificar acá?** Ninguno. El tamaño es proporcional a la frecuencia,
pero no hay escala ni eje.

**¿Qué decide la posición y el color?** El azar. `random_state=42` lo fija para que la
imagen salga siempre igual; cambiá el número y la nube se reordena entera sin que ningún
dato haya cambiado.

**Pero eso no es culpa de la nube: es culpa de haber hecho una sola.** Doce nubes, una por
año, con la misma configuración, ya responden una pregunta: *¿cambia lo que escribe?*

In [ ]:
# Con diez palabras por año y el color según el puesto, el color deja de ser decorativo.
def color_por_puesto(palabra, **argumentos_que_ignoramos):
    """Devuelve el color de la rampa que le toca a la palabra según su puesto."""
    puesto = puestos_de_la_nube.get(palabra, len(RAMPA) - 1)
    paso = puesto * len(RAMPA) // 10
    if paso > len(RAMPA) - 1:
        paso = len(RAMPA) - 1
    return RAMPA[paso]


print("función de color lista: el índigo es el puesto 1, el rosa el puesto 10")

In [ ]:
def nube_de_un_anio(conteo_del_anio, cuantas=10):
    """Arma la nube de un año y deja anotado el puesto de cada palabra."""
    # 'puestos_de_la_nube' es global a propósito: color_por_puesto la consulta.
    global puestos_de_la_nube
    puestos_de_la_nube = {}
    frecuencias = {}

    puesto = 0
    for palabra, veces in conteo_del_anio.most_common(cuantas):
        puestos_de_la_nube[palabra] = puesto
        frecuencias[palabra] = veces
        puesto = puesto + 1

    nube = WordCloud(width=520, height=300, background_color=SUPERFICIE,
                     max_words=cuantas, random_state=SEMILLA,
                     font_path=RUTA_FUENTE, color_func=color_por_puesto)
    nube.generate_from_frequencies(frecuencias)
    return nube

In [ ]:
figura, paneles = plt.subplots(4, 3, figsize=(13.5, 13))
figura.patch.set_facecolor(SUPERFICIE)
lista_de_paneles = paneles.flatten()

posicion = 0
for anio in anios:
    panel = lista_de_paneles[posicion]
    panel.imshow(nube_de_un_anio(conteos_por_anio[anio]), interpolation="bilinear")
    panel.axis("off")
    panel.grid(False)
    panel.set_title(str(anio), color=TINTA)
    posicion = posicion + 1

plt.tight_layout()
plt.show()

Mirá las doce con atención antes de festejar.

**En ocho de los doce años, las diez palabras más frecuentes son `dos`, `vez`, `años`,
`había`, `ahora`, `después`, `siempre`.** Son exactamente las candidatas que detectamos en
la sección 6. Las nubes no están mostrando los temas de cada año: están mostrando **la
lista de palabras vacías que todavía no aplicamos**.

Los años donde sí se ve algo son los que tienen una palabra lo bastante insistente como
para ganarle al ruido: `alex` y `lucas` en 2004, `revista`, `orsai`, `chiri`, `número` y
`lectores` entre 2010 y 2012, `papelitos` en 2013, `hija` en 2015.

Probemos entonces lo que quedó pendiente en la sección 6: **las mismas doce nubes, sin las
veintidós candidatas.**

Sigue sin haber escala —eso no se arregla— pero la comparación entre paneles hace un
trabajo que una nube sola no podía hacer.

Para las palabras que ya sabemos que importan, el gráfico honesto es otro: una línea por
palabra, **normalizada**, con eje y con números.

In [ ]:
# La lista base más las veintidós candidatas de la sección 6.
palabras_vacias_ampliadas = palabras_vacias + list(tabla_candidatas["palabra"])

conteos_sin_candidatas = {}
for anio in anios:
    conteos_sin_candidatas[anio] = contar_con_contenido(palabras_por_anio[anio],
                                                        palabras_vacias_ampliadas)

print("palabras vacías en la lista base    :", len(palabras_vacias))
print("palabras vacías en la lista ampliada:", len(palabras_vacias_ampliadas))

In [ ]:
figura, paneles = plt.subplots(4, 3, figsize=(13.5, 13))
figura.patch.set_facecolor(SUPERFICIE)
lista_de_paneles = paneles.flatten()

posicion = 0
for anio in anios:
    panel = lista_de_paneles[posicion]
    panel.imshow(nube_de_un_anio(conteos_sin_candidatas[anio]), interpolation="bilinear")
    panel.axis("off")
    panel.grid(False)
    panel.set_title(str(anio), color=TINTA)
    posicion = posicion + 1

plt.tight_layout()
plt.show()

Otras doce nubes, los mismos datos, **veintidós palabras menos**.

Ahora hay años que dicen algo: `alex` y `lucas` en 2004 y 2005, `andrés`, `gelós` y `pablo`
en 2009, `revista`, `orsai`, `lectores` y `número` entre 2010 y 2012, `papelitos` y `pepe`
en 2013, `fútbol` y `mundial` en 2014, `hija` y `libros` en 2015.

Pero pasaron dos cosas más, y ninguna es menor.

**Primero: apareció una capa nueva de ruido.** `cosas`, `gente`, `bien`, `nadie`, `así`,
`tarde`, `noche`, `veces`, `menos`. Sacar veintidós palabras destapa las veintidós
siguientes. Una lista de palabras vacías **no termina nunca**: se corta en algún punto, y
ese punto lo elegís vos.

**Segundo, y peor: `chiri` no está en ninguna de las doce nubes.** Estaba en la lista de
candidatas y esta vez la aplicamos entera. El coautor del corpus desapareció del gráfico
por una decisión que tomamos tres secciones atrás, y en la imagen no queda ningún rastro
de que eso pasó.

> **Las dos grillas son ciertas.** La primera muestra qué palabras dice más el corpus; la
> segunda, qué palabras dice más *una vez que decidimos qué no cuenta*. La diferencia entre
> las dos imágenes es una decisión nuestra, y no está escrita en ningún lugar del gráfico.

Para las palabras que ya sabemos que importan, el gráfico honesto es otro: una línea por
palabra, **normalizada**, con eje y con números.

In [ ]:
palabras_a_seguir = ["orsai", "revista", "lucas", "padre", "hija", "chiri"]

evolucion = pd.DataFrame({"anio": anios})
for palabra in palabras_a_seguir:
    tabla_de_la_palabra = frecuencias_de(palabras_por_anio, palabra)
    evolucion[palabra] = tabla_de_la_palabra["por_10k"].values

evolucion

In [ ]:
plt.figure(figsize=(9, 4.4))

colores = [VIOLETA, ROSA, RAMPA[0], RAMPA[2], NEUTRO, RAMPA[4]]
posicion = 0
for palabra in palabras_a_seguir:
    plt.plot(evolucion["anio"], evolucion[palabra], marker="o", markersize=4,
             linewidth=1.8, color=colores[posicion], label=palabra)
    posicion = posicion + 1

plt.ylabel("apariciones por cada 10.000 palabras")
plt.title("Seis palabras a lo largo de doce años")
plt.legend(frameon=False, ncol=3)
plt.tight_layout()
plt.show()

Cuatro lecturas que ahora se pueden verificar contra la tabla:

- **`revista` pasa de menos de 3,4 por diez mil a 61,1 en 2010.** Es el salto más grande
  del corpus.
- **`orsai` no arranca en cero**: está entre 2,2 y 8,0 desde 2004, porque así se llamaba el
  blog. Lo que cambia en 2010 es la escala: sube a 25,8 y llega a 39,4 en 2012.
- **`lucas` hace el camino inverso**: 29,9 en 2004, cero en 2009 y en 2013, y **16,8 en
  2014**, cuando Casciari vuelve al teatro.
- **`hija` salta a 22,5 en 2015**, cinco veces más que en cualquier año anterior.

**Esto sí es una afirmación verificable**: hay eje, hay escala y los números están arriba.

### ✎ Para pensar

- Si mirabas solo las nubes, ¿podías afirmar que `orsai` ya estaba en 2004? ¿Y con este
  gráfico?
- La línea de `lucas` vuelve a subir en 2014. ¿Qué habría que leer para saber si es el
  mismo Lucas?

## 10. Etapa cinco: el retorno al texto

Hasta acá todo fue en una dirección: texto, palabras, cuentas, matrices, gráficos. Cada
etapa dejó atrás algo del texto original, y a esta altura estamos mirando números que ya
no se parecen a nada que alguien haya escrito.

**La operación que falta va en la dirección contraria.**

Una **concordancia** es todas las apariciones de un término, con su cotexto alrededor. El
término buscado se llama **nodo** y acá va entre corchetes.

### Es la operación más vieja de la disciplina

En 1946 el jesuita **Roberto Busa** planeó un índice de toda la obra de Tomás de Aquino.
En 1949 consiguió que Thomas J. Watson lo financiara con IBM. Son 10.631.980 palabras de
latín medieval, pasadas a tarjetas perforadas. El primer volumen salió en 1974 y la obra
se completó en 56 tomos en 1980: **treinta y cuatro años de trabajo**.

El *Index Thomisticus* es una concordancia, y se lo considera el acto fundacional de las
humanidades computacionales. **Lo primero que se hizo con una computadora sobre un texto
fue esto.**

El nombre corto, *KWIC*, lo acuñó Hans Peter Luhn en IBM en 1958. La idea manual es
anterior: el bibliotecario Andrea Crestadoro la propuso entre 1856 y 1864.

**Dónde vive hoy.** En todos los concordanciadores de lingüística de corpus (AntConc,
Sketch Engine), en la búsqueda de tu editor de texto, en el `grep` de la terminal, y en
cualquier herramienta legal o periodística que muestre "coincidencias en contexto".

In [ ]:
def concordancia_de(corpus, termino, ventana=45):
    """Todas las apariciones de un término, con su cotexto a cada lado.

    Busca sobre el texto ORIGINAL, no sobre el limpio: la limpieza destruye
    justamente la puntuación y las mayúsculas que hacen legible una cita.
    """
    patron = r"\b" + re.escape(termino.lower()) + r"\b"

    filas = []
    for fila in corpus.itertuples():
        for hallazgo in re.finditer(patron, fila.texto.lower()):
            desde = hallazgo.start()
            hasta = hallazgo.end()
            filas.append({
                "anio": fila.anio,
                "izquierda": fila.texto[max(0, desde - ventana):desde],
                "nodo": "[" + fila.texto[desde:hasta] + "]",
                "derecha": fila.texto[hasta:hasta + ventana],
            })
    return pd.DataFrame(filas)

Probémosla con `che`, el marcador oral rioplatense por excelencia.

In [ ]:
concordancia_che = concordancia_de(corpus, "che", ventana=45)

print("apariciones de 'che':", len(concordancia_che))
concordancia_che.head(8)

### Leé la columna del medio antes de sacar conclusiones

Veinte apariciones en doce años y 421.000 palabras. Un conteo se detendría acá, y la
conclusión saldría sola: *el autor casi no usa el marcador oral rioplatense*.

Mirá las filas.

- *«Che, Lucas, ¿vamos a comer barro?»* — la interjección.
- *«también al Che y a Gardel»* — **Ernesto Guevara.**
- *«los interminables discursos de un Che Guevara viejo»* — Guevara otra vez.
- *«Maradona, el Che, Eva Perón, Charly, Borges, Monzón»* — Guevara de nuevo.

**Seis de las veinte no son la interjección: son una persona.** El conteo sobreestima el
marcador oral en un treinta por ciento, y ninguna cuenta de frecuencias podía avisarte,
porque las dos cosas se escriben igual.

Esto tiene nombre —dos palabras distintas con la misma forma escrita— y no se resuelve
contando mejor. Se resuelve mirando.

> ### Perilla
>
> `concordancia_de` tiene un parámetro `ventana`, que por omisión vale 45 caracteres.
>
> Corré la celda siguiente y compará las tres ventanas.
>
> - Con 15, ¿podés distinguir el *che* interjección del *Che* persona?
> - Con 120, ¿pasa lo que esperabas?

In [ ]:
for ventana in [15, 45, 120]:
    prueba = concordancia_de(corpus, "che", ventana=ventana)
    fila = prueba.iloc[2]
    print("ventana =", ventana)
    print("   ", fila["izquierda"], fila["nodo"], fila["derecha"])
    print()

Con 120 el cotexto **aparece cortado igual**, con puntos suspensivos, si mirás la tabla en
vez del `print`. No es un error de la función: pandas trunca las celdas al mostrarlas.

In [ ]:
print("ancho máximo de celda que usa pandas:", pd.get_option("display.max_colwidth"))

# Lo subimos para poder leer la concordancia como tabla.
pd.set_option("display.max_colwidth", 130)

concordancia_de(corpus, "che", ventana=60).head(4)

Cincuenta caracteres, y no lo avisa nadie.

O sea que acá hay **dos perillas, y solo una la elegiste vos.** Si no supieras de la
segunda, dirías que tu función anda mal.

> **La ventana no es una preferencia visual: decide qué podés ver.** Es una perilla de
> método, como el nivel de agregación o la lista de palabras vacías. Va anotada, y la de
> la librería también.

### Y algo que no fuimos a buscar

Volvé a mirar la concordancia de `che`. La fila de *«Maradona, el Che, Eva Perón»* aparece
**dos veces**: una en 2007 y otra en 2012.

No es un error de la función. Es el mismo párrafo, palabra por palabra, en dos años
distintos. Comprobémoslo sobre todo el corpus.

In [ ]:
conteo_de_parrafos = Counter(corpus["texto"])

parrafos_repetidos = 0
filas_de_mas = 0
for texto in conteo_de_parrafos:
    veces = conteo_de_parrafos[texto]
    if veces > 1:
        parrafos_repetidos = parrafos_repetidos + 1
        filas_de_mas = filas_de_mas + veces - 1

print("párrafos que aparecen más de una vez:", parrafos_repetidos)
print("filas de más por duplicación        :", filas_de_mas)

Doce párrafos repetidos y catorce filas de más. Veamos cuáles son.

In [ ]:
for texto, veces in conteo_de_parrafos.most_common(3):
    print(veces, "veces:", texto[:100], "...")
    print()

El más repetido aparece cuatro veces y **no es un texto**: es un pie de página que el autor
pegaba al final de sus entradas. Entró al corpus como si fuera contenido.

Doce párrafos duplicados sobre 5.428 son el 0,2 %. Poco. Pero cada uno suma dos, tres o
cuatro veces en todos los conteos que hiciste en el práctico, y el pie de página infla las
frecuencias de *Orsai*, *relatos*, *audio* y *blog* sin que nadie lo haya decidido.

**Lo encontramos buscando otra cosa.** Eso es lo que hace una concordancia: te obliga a
mirar los casos, y los casos tienen cosas que las cuentas no.

> Ninguno de estos dos hallazgos hace saltar un error. Los dos devuelven números con buena
> cara.

### ✎ Para pensar

- ¿Qué habría que hacer con los doce párrafos duplicados? ¿Borrarlos, dejarlos o
  documentarlos?
- El pie de página nombra la revista. ¿En qué años esperás que esté, y qué le hace eso a
  la línea de `orsai` de la sección 9?

## 11. Etapa seis: del texto a un signo

**La analogía.** Un lexicón afectivo es un diccionario donde cada palabra tiene, además de
su significado, un signo: esta empuja hacia lo agradable, esta otra hacia lo desagradable.
Analizar el sentimiento de un texto con un lexicón es **pasar el texto por ese diccionario
y sumar**.

**Dónde vive esto.** Es una de las tareas más vendidas del procesamiento de lenguaje:
monitoreo de marcas, análisis de reseñas, encuestas abiertas, moderación. También es una
de las que más seguido se aplica mal, y este cuaderno va a mostrar exactamente cómo.

### Lo que hace el ejercicio original, y por qué no lo repetimos

El material del que viene esta unidad resuelve el sentimiento así:

```python
traducir = lambda x: TextBlob(x).translate(to="en")
data['polarity'] = data['blob_en'].apply(lambda x: x.sentiment.polarity)
```

Traduce cada texto al inglés y le pide la polaridad a TextBlob. Tres problemas:

| Problema | Por qué importa acá |
|---|---|
| Hoy no corre | La traducción de TextBlob dependía de una API que dejó de responder |
| Traduce sin auditoría | *«Che, boluuudo»* pasa por un traductor automático y llega irreconocible |
| No podés ver la cuenta | La polaridad sale de un modelo entrenado en reseñas en inglés |

Vamos a hacerlo con un **lexicón en español, escrito a mano, que está en un archivo que
podés abrir y discutir**. Va a andar peor que un modelo grande. Pero vas a poder ver por
qué se equivoca, que es el punto del cuaderno.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Lexicón afectivo** | Lista de palabras con un valor de polaridad asignado |
| **Polaridad** | Si algo empuja hacia lo positivo (+) o lo negativo (−), y cuánto |
| **Densidad afectiva** | Qué proporción de las palabras del texto está en el lexicón |
| **Negador** | Palabra que da vuelta el signo de lo que viene después: *no*, *nunca* |
| **Ventana de negación** | Cuántas palabras hacia atrás se busca un negador |

### Primero, abrí el archivo

`datos/lexicon_afectivo_es.csv` empieza con treinta líneas de comentario que declaran quién
lo armó, con qué criterio y **qué límites tiene**. Abrilo antes de correr la celda
siguiente: no hay nada mágico adentro.

In [ ]:
# 'comment="#"' le dice a pandas que las líneas que empiezan con # no son datos.
lexicon = pd.read_csv(CARPETA_ANEXO / "lexicon_afectivo_es.csv", comment="#")

print("entradas en el lexicón:", len(lexicon))
print()
print(lexicon["polaridad"].value_counts().sort_index())
lexicon.head(4)

291 palabras, de −2 a +2. Y cuatro en **cero**, que están a propósito.

In [ ]:
en_cero = lexicon[lexicon["polaridad"] == 0]

print("las palabras que no se pudieron puntuar:")
print(list(en_cero["palabra"]))

`increíble`, `loco`, `tremendo`, `bárbaro`. En rioplatense las cuatro pueden ser un elogio
o un insulto según el contexto, y **una palabra solo puede tener un puntaje**. Ese es el
límite central del método, declarado desde el archivo.

Ahora la función que puntúa.

In [ ]:
# Un diccionario palabra -> polaridad es mucho más rápido de consultar que una tabla.
polaridad_de_palabra = {}
for fila in lexicon.itertuples():
    polaridad_de_palabra[fila.palabra] = fila.polaridad

# Los negadores dan vuelta el signo de lo que viene después.
NEGADORES = ["no", "nunca", "nada", "ni", "jamás", "tampoco"]

# Cuántas palabras hacia atrás buscamos un negador. Es una perilla: probala en 1 y en 6.
VENTANA_DE_NEGACION = 3

print("negadores:", NEGADORES)
print("ventana de negación:", VENTANA_DE_NEGACION, "palabras hacia atrás")

In [ ]:
def puntuar_texto(texto_limpio, polaridad_de_palabra, con_negacion=True):
    """Suma la polaridad de las palabras del texto que están en el lexicón.

    Recibe el lexicón como diccionario: así se puede correr la misma función con
    dos lexicones distintos y comparar, que es lo que vamos a hacer más abajo.

    Devuelve un diccionario con la suma, cuántas palabras afectivas encontró y
    cuántas palabras tiene el texto en total.
    """
    palabras = texto_limpio.split()
    suma = 0
    afectivas = 0
    posicion = 0

    for palabra in palabras:
        if palabra in polaridad_de_palabra:
            valor = polaridad_de_palabra[palabra]
            if con_negacion:
                desde = max(0, posicion - VENTANA_DE_NEGACION)
                for anterior in palabras[desde:posicion]:
                    if anterior in NEGADORES:
                        valor = -valor
                        break
            suma = suma + valor
            afectivas = afectivas + 1
        posicion = posicion + 1

    return {"suma": suma, "afectivas": afectivas, "palabras": len(palabras)}

Probémosla en tres oraciones escritas para la ocasión —como las cuatro del teórico— antes
de soltarla sobre el corpus.

In [ ]:
oraciones_de_prueba = [
    "fue una tarde hermosa y llena de risas con mis amigos",
    "no fue una tarde hermosa: fue un desastre triste",
    "el tren llega a las ocho y media desde la estación de plaza constitución",
]

for oracion in oraciones_de_prueba:
    resultado = puntuar_texto(oracion, polaridad_de_palabra)
    print(resultado, "<-", oracion)

La primera suma positivo. La segunda arranca con `no`, que da vuelta a `hermosa`, y suma
negativo. La tercera **no tiene ninguna palabra del lexicón**: suma cero, y ese cero no
quiere decir *neutral*, quiere decir **no sé**.

Confundir esas dos cosas es el error más común de todo este método. Guardá esa distinción.

In [ ]:
sumas = []
afectivas = []
largos = []
for texto in corpus["texto_limpio"]:
    resultado = puntuar_texto(texto, polaridad_de_palabra)
    sumas.append(resultado["suma"])
    afectivas.append(resultado["afectivas"])
    largos.append(resultado["palabras"])

corpus["suma_afectiva"] = sumas
corpus["palabras_afectivas"] = afectivas
corpus["palabras"] = largos

assert corpus["palabras_afectivas"].sum() > 0, "El lexicón no encontró ni una palabra."

cantidad_de_parrafos = len(corpus)
print("párrafos puntuados:", f"{cantidad_de_parrafos:,}")
corpus[["anio", "suma_afectiva", "palabras_afectivas", "palabras"]].head(3)

### La cobertura, antes que cualquier resultado

La primera pregunta no es *qué sentimiento tiene el corpus*. Es **cuánto del corpus llegó
a ver el lexicón.**

In [ ]:
afectivas_totales = corpus["palabras_afectivas"].sum()
palabras_totales = corpus["palabras"].sum()
sin_ninguna = corpus["palabras_afectivas"] == 0
cuantos_sin_ninguna = sin_ninguna.sum()

cobertura = 100 * afectivas_totales / palabras_totales
proporcion_sin_afecto = 100 * cuantos_sin_ninguna / len(corpus)

print("palabras del corpus vistas por el lexicón:", f"{afectivas_totales:,}")
print("sobre un total de                        :", f"{palabras_totales:,}")
print("cobertura                                :", f"{cobertura:.2f}%")
print()
print("párrafos sin NINGUNA palabra afectiva    :", f"{cuantos_sin_ninguna:,}")
print("o sea el                                 :", f"{proporcion_sin_afecto:.0f}% del corpus")

**El lexicón ve el 1,5 % de las palabras, y casi la mitad de los párrafos le resultan
invisibles.**

Ese número tiene que ir en cualquier informe que use este método, y casi nunca va. Con 291
palabras contra un vocabulario de 30.267, no podía ser de otra manera.

**Consecuencia práctica:** no se puede promediar la polaridad de los párrafos, porque 2.529
de ellos aportarían un cero que significa *no sé*. Agregamos distinto: **sumamos por año y
dividimos por la cantidad de palabras afectivas de ese año.** Así, cada palabra afectiva
pesa igual y los párrafos mudos no votan.

In [ ]:
filas = []
for anio in anios:
    del_anio = corpus[corpus["anio"] == anio]
    suma_del_anio = del_anio["suma_afectiva"].sum()
    afectivas_del_anio = del_anio["palabras_afectivas"].sum()
    palabras_del_anio = del_anio["palabras"].sum()
    filas.append({
        "anio": anio,
        "polaridad": round(suma_del_anio / afectivas_del_anio, 3),
        "densidad_afectiva": round(100 * afectivas_del_anio / palabras_del_anio, 2),
        "palabras_afectivas": afectivas_del_anio,
    })

sentimiento_por_anio = pd.DataFrame(filas)
sentimiento_por_anio

In [ ]:
plt.figure(figsize=(9, 4))

colores_de_las_barras = []
for valor in sentimiento_por_anio["polaridad"]:
    if valor < 0:
        colores_de_las_barras.append(VIOLETA)
    else:
        colores_de_las_barras.append(ROSA)

plt.bar(sentimiento_por_anio["anio"], sentimiento_por_anio["polaridad"],
        color=colores_de_las_barras)
plt.axhline(y=0, color=TINTA, linewidth=1)
plt.ylabel("polaridad por palabra afectiva")
plt.title("Polaridad del corpus por año, según el lexicón de la cátedra")
plt.tight_layout()
plt.show()

**Ocho años en negativo y cuatro seguidos en positivo: 2010, 2011, 2012 y 2013.** El pico
es 2010, con +0,319; el fondo es 2009, con −0,267.

Antes de contar una historia con esto, la otra mitad del gráfico: el artículo original
cruza polaridad con subjetividad. Nosotros cruzamos polaridad con **densidad afectiva**,
que es lo que nuestro método puede medir de verdad.

In [ ]:
plt.figure(figsize=(7.5, 4.6))
plt.scatter(sentimiento_por_anio["densidad_afectiva"], sentimiento_por_anio["polaridad"],
            color=VIOLETA, s=70, zorder=3)
plt.axhline(y=0, color=NEUTRO, linewidth=1)

for fila in sentimiento_por_anio.itertuples():
    plt.annotate(str(fila.anio), xy=(fila.densidad_afectiva, fila.polaridad),
                 xytext=(6, -3), textcoords="offset points", fontsize=9, color=TINTA)

plt.xlabel("densidad afectiva: % de palabras que el lexicón reconoce")
plt.ylabel("polaridad")
plt.title("Cada año, en las dos dimensiones que el lexicón puede medir")
plt.tight_layout()
plt.show()

Los cuatro años positivos son, además, los de **menor densidad afectiva** (1,2 % a 1,3 %,
contra 1,65 % a 2,04 % del resto). Es decir: en los años que salen "más positivos" el
lexicón reconoce **menos** palabras.

Eso ya debería encender una alarma. Un resultado que aparece justo donde hay menos
evidencia no es un hallazgo: es un candidato a artefacto.

### La auditoría: quién empuja el promedio

Un promedio no dice de dónde vino. Preguntémosle.

In [ ]:
aporte_por_palabra = Counter()
for texto in corpus["texto_limpio"]:
    for palabra in texto.split():
        if palabra in polaridad_de_palabra:
            aporte_por_palabra[palabra] += polaridad_de_palabra[palabra]

mas_positivas = aporte_por_palabra.most_common(8)
todas_ordenadas = aporte_por_palabra.most_common()
mas_negativas = todas_ordenadas[-8:]

print("las 8 que más empujan hacia lo positivo:")
for palabra, aporte in mas_positivas:
    print("  ", palabra, "->", aporte)
print()
print("las 8 que más empujan hacia lo negativo:")
for palabra, aporte in mas_negativas:
    print("  ", palabra, "->", aporte)

`amor`, `amigos`, `feliz`, `amigo` arriba; `muerte`, `peor`, `muerto`, `miedo` abajo. Para
un corpus autobiográfico rioplatense, es plausible.

Pero en el quinto puesto de lo positivo está **`bueno`, con 148 apariciones**. Y `bueno`,
en rioplatense, casi nunca es un adjetivo de aprobación: es un **marcador del discurso**,
el equivalente de un carraspeo. *"Bueno, che, dale."*

No lo discutamos. **Vayamos a leerlo**, que para eso escribimos la concordancia.

In [ ]:
concordancia_bueno = concordancia_de(corpus, "bueno", ventana=40)

print("apariciones de 'bueno':", len(concordancia_bueno))
concordancia_bueno.head(6)

Se ve a simple vista en las seis primeras filas. Contémoslo sobre las 148.

In [ ]:
# Un marcador del discurso arranca el turno de habla: viene al principio del párrafo,
# después de un guion de diálogo o de dos puntos, y casi siempre lo sigue una coma.
usos_como_marcador = 0
for fila in concordancia_bueno.itertuples():
    izquierda = fila.izquierda.strip()
    derecha = fila.derecha.strip()
    if derecha.startswith(",") or izquierda == "" or izquierda.endswith(("—", "-", ":", "«", '"')):
        usos_como_marcador = usos_como_marcador + 1

print("apariciones de 'bueno' que son marcador del discurso:", usos_como_marcador)
print("sobre un total de                                   :", len(concordancia_bueno))

**58 de 148.** Casi cuatro de cada diez `bueno` del corpus no expresan nada positivo: son
un ruido de la conversación que el lexicón está contando como emoción.

Y hay más: mirá `suerte`, en el sexto puesto. En este corpus aparece casi siempre en *"por
suerte no..."*, donde el negador ya le da vuelta el signo, y en *"la suerte de"*, donde no
es una emoción sino el destino.

### ✎ Antes de ejecutar

Saquemos `bueno` del lexicón y recalculemos. Es una palabra sobre 291, y el 1,5 % de
cobertura no cambia casi nada. ¿Esperás que algún año cambie de signo?

In [ ]:
# Una copia del lexicón sin esa palabra. El original queda intacto.
lexicon_sin_bueno = dict(polaridad_de_palabra)
del lexicon_sin_bueno["bueno"]

filas = []
for anio in anios:
    del_anio = corpus[corpus["anio"] == anio]
    suma = 0
    afectivas_del_anio = 0
    for texto in del_anio["texto_limpio"]:
        resultado = puntuar_texto(texto, lexicon_sin_bueno)
        suma = suma + resultado["suma"]
        afectivas_del_anio = afectivas_del_anio + resultado["afectivas"]
    filas.append({"anio": anio, "sin_bueno": round(suma / afectivas_del_anio, 3)})

comparacion = sentimiento_por_anio[["anio", "polaridad"]].merge(pd.DataFrame(filas), on="anio")
comparacion["cambio_de_signo"] = np.sign(comparacion["polaridad"]) != np.sign(comparacion["sin_bueno"])
comparacion

**2012 cambia de signo.** Pasa de +0,020 a −0,021 porque saqué **una palabra** de una lista
de 291.

Ese año, en el gráfico de barras, estaba del lado positivo. Ahora está del negativo. Si
alguien hubiera escrito *"a partir de 2010 el tono se vuelve positivo y se mantiene cuatro
años"*, esa afirmación se acaba de romper con una línea de código.

> **La regla que queda.** Un resultado que se da vuelta cuando cambiás una palabra del
> lexicón no es un resultado: es una decisión tuya disfrazada de medición. Antes de
> publicar cualquier número de sentimiento, sacale las tres palabras que más aportan y
> mirá si sobrevive.

> ### Perilla
>
> `puntuar_texto` tiene la perilla `con_negacion`, y arriba está `VENTANA_DE_NEGACION`.
>
> La celda siguiente apaga la negación. Antes de correrla: ¿esperás que el corpus se vuelva
> más positivo o más negativo cuando *dejamos* de dar vuelta los signos?

In [ ]:
suma_sin_negacion = 0
afectivas_sin_negacion = 0
parrafos_que_cambian = 0

for texto in corpus["texto_limpio"]:
    con = puntuar_texto(texto, polaridad_de_palabra, con_negacion=True)
    sin = puntuar_texto(texto, polaridad_de_palabra, con_negacion=False)
    suma_sin_negacion = suma_sin_negacion + sin["suma"]
    afectivas_sin_negacion = afectivas_sin_negacion + sin["afectivas"]
    if con["suma"] != sin["suma"]:
        parrafos_que_cambian = parrafos_que_cambian + 1

polaridad_con = corpus["suma_afectiva"].sum() / corpus["palabras_afectivas"].sum()
polaridad_sin = suma_sin_negacion / afectivas_sin_negacion

print("polaridad global CON negación:", f"{polaridad_con:.4f}")
print("polaridad global SIN negación:", f"{polaridad_sin:.4f}")
print("párrafos donde la negación cambia algo:", parrafos_que_cambian)

Sin negación el corpus se ve **más negativo** (−0,144 contra −0,119), y la diferencia sale
de 284 párrafos: un 5 % del corpus.

¿Por qué más negativo? Porque en este corpus el patrón dominante es *"por suerte **no**
pasó nada"*, *"**no** hay peligro"*: negaciones sobre palabras negativas que el modelo, sin
la regla, cuenta como si el mal hubiera ocurrido.

**Tres líneas de código —una ventana de tres palabras— movieron el resultado más que
cualquier decisión de las secciones 1 a 9.**

### Qué medimos en realidad

Volvamos a los cuatro años positivos: 2010, 2011, 2012 y 2013. Son exactamente los años en
que, según la sección 7, el vocabulario característico pasó de los personajes de los
relatos a `libreros`, `revista`, `número`, `suscripción`.

Ahí está la explicación, y no es emocional: **los textos de esos años no son cuentos, son
comunicaciones de una editorial.** Hablan de logística, de tiradas y de lectores. Tienen
menos palabras afectivas (menor densidad) y las que tienen son de agradecimiento.

> El análisis de sentimiento no midió el ánimo del autor. **Midió que el autor cambió de
> tema.** Y eso solo se puede saber cruzándolo con otra medición y yendo a leer los casos.

### ✎ Para pensar

- Si tuvieras que reportar este análisis en un informe, ¿qué tres números pondrías al lado
  de cada polaridad para que nadie la lea de más?
- El lexicón tiene 291 palabras y el corpus 30.267 distintas. ¿Cuál sería el próximo paso
  para mejorar la cobertura sin escribir 30.000 líneas a mano?

## 12. Modelado de tópicos

**La analogía.** Imaginate una cocina donde cada plato salió de mezclar en distintas
proporciones un puñado de recetas base. Tenés los platos, no tenés las recetas. El
modelado de tópicos es el procedimiento para **reconstruir las recetas mirando los platos**.

Cada párrafo del corpus se supone una mezcla de unos pocos temas; cada tema, una mezcla de
palabras. El algoritmo busca esas dos mezclas a la vez.

**Dónde vive esto.** Clasificación automática de reclamos, exploración de archivos
históricos, análisis de respuestas abiertas de encuestas, y todo caso donde hay demasiados
documentos para leerlos y hace falta un mapa antes de elegir qué leer.

### Por qué NMF y no LDA

El ejercicio original usa **LDA** de gensim. Nosotros usamos **NMF** de scikit-learn, por
tres razones que tienen que ver con este corpus y con esta materia:

| | LDA | NMF |
|---|---|---|
| Qué necesita | conteos crudos | una matriz TF-IDF |
| Reproducibilidad | muestrea: dos corridas dan resultados distintos | determinista |
| Librería nueva | sí, gensim | no: `sklearn` ya está |

TF-IDF es lo que la unidad enseñó en `teorico/02`, y el determinismo importa en una clase:
si el resultado cambia entre dos corridas, nadie puede saber si cambió porque tocó un
parámetro o porque el azar se movió.

### Y la decisión previa: qué es un documento

En la sección 5 vectorizamos **doce documentos**, uno por año. Para tópicos eso no sirve:
doce documentos enormes, cada uno con todos los temas adentro. No hay nada que separar.

Acá la unidad tiene que ser el **párrafo**. Son 5.428 documentos, cada uno lo bastante
corto para hablar de una cosa.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 'min_df=5'   descarta lo que aparece en menos de 5 párrafos: saca erratas y nombres únicos.
# 'max_df=0.5' descarta lo que aparece en más de la mitad de los párrafos: ahí no hay tema.
vectorizador = TfidfVectorizer(stop_words=palabras_vacias, min_df=5, max_df=0.5)
pesos_por_parrafo = vectorizador.fit_transform(corpus["texto_limpio"])
vocabulario = vectorizador.get_feature_names_out()

celdas = pesos_por_parrafo.shape[0] * pesos_por_parrafo.shape[1]
print("forma de la matriz:", pesos_por_parrafo.shape)
print("porcentaje lleno  :", f"{100 * pesos_por_parrafo.nnz / celdas:.2f}%")

**0,44 % lleno**, contra el 36,5 % de la matriz por años. Es lo que anticipamos en la sección
5: cambiar la unidad de análisis cambia la forma de la matriz.

Con esta matriz `.toarray()` pediría casi 300 MB. No lo hagamos.

In [ ]:
from sklearn.decomposition import NMF

# 'n_components' es cuántos tópicos buscamos. Es LA perilla: la elegimos nosotros.
CUANTOS_TOPICOS = 4

# 'init' e 'max_iter' no cambian el análisis: aseguran un arranque fijo y que converja.
modelo = NMF(n_components=CUANTOS_TOPICOS, random_state=SEMILLA,
             init="nndsvd", max_iter=400)
reparto_por_parrafo = modelo.fit_transform(pesos_por_parrafo)

print("tópicos buscados       :", CUANTOS_TOPICOS)
print("un peso por párrafo y tópico:", reparto_por_parrafo.shape)
print("y un peso por palabra y tópico:", modelo.components_.shape)

In [ ]:
def palabras_del_topico(componente, vocabulario, cuantas=10):
    """Las palabras con más peso en un tópico, de mayor a menor."""
    orden = np.argsort(componente)
    mejores = []
    for indice in orden[-cuantas:]:
        mejores.append(vocabulario[indice])
    mejores.reverse()
    return mejores


filas = []
numero_de_topico = 0
for componente in modelo.components_:
    palabras = palabras_del_topico(componente, vocabulario)
    filas.append({"topico": numero_de_topico, "palabras": ", ".join(palabras)})
    numero_de_topico = numero_de_topico + 1

pd.DataFrame(filas)

### Ponerles nombre es trabajo humano

El algoritmo devolvió cuatro listas de palabras. **No devolvió ningún tema**: los temas los
lees vos.

| Tópico | Palabras | Qué es |
|---|---|---|
| 0 | había, casa, años, dos, después, dijo, chiri, tarde | El relato en pasado |
| 1 | lucas, alex, vos, vieja, tenés, pito, vas, alfonso | Las obras de teatro |
| 2 | orsai, revista, número, lectores, libro, ustedes, blog | La revista Orsai |
| 3 | si, siempre, ser, ahora, mundo, ha, mismo, nunca | **No es un tema** |

Dos de los cuatro son temas de verdad. El 0 es una manera de narrar más que un asunto. Y
el 3 no es nada: son las palabras que sobraron, las que están en todo el corpus y no
alcanzaron a caer en ningún otro lado.

> **Eso es lo normal, y casi nunca se cuenta.** Un modelo de k tópicos siempre devuelve k
> listas, tenga o no k temas adentro. La cantidad la elegiste vos; el modelo obedece.

### ✎ Antes de ejecutar

Si el tópico 1 son las obras de teatro con Lucas y Alex, y el 2 es la revista Orsai:
¿en qué años esperás encontrar cada uno? Escribilo antes de mirar la tabla.

In [ ]:
# A cada párrafo le asignamos el tópico en el que pesa más.
corpus["topico"] = np.argmax(reparto_por_parrafo, axis=1)

# 'crosstab' cuenta cuántos párrafos hay de cada combinación año-tópico.
# 'normalize="index"' convierte cada fila en porcentajes de ese año.
reparto_por_anio = pd.crosstab(corpus["anio"], corpus["topico"], normalize="index")
(100 * reparto_por_anio).round(1)

In [ ]:
plt.figure(figsize=(9.5, 4.2))

nombres = ["0 · relato en pasado", "1 · obras de teatro", "2 · revista Orsai", "3 · resto"]
colores = [NEUTRO, VIOLETA, ROSA, "#d8d5de"]
base = np.zeros(len(anios))

numero_de_topico = 0
for nombre in nombres:
    altura = 100 * reparto_por_anio[numero_de_topico].values
    plt.bar(anios, altura, bottom=base, color=colores[numero_de_topico], label=nombre)
    base = base + altura
    numero_de_topico = numero_de_topico + 1

plt.ylabel("% de los párrafos del año")
plt.title("De qué habla cada año, según el modelo")
plt.legend(frameon=False, fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

**La revista Orsai pasa del 5-10 % antes de 2010 al 53 % en 2010 y 2011.** Las obras de
teatro son el 20,5 % de 2004, desaparecen entre 2009 y 2013, y **vuelven con el 8,2 % en
2014**: lo mismo que ya había mostrado la línea de `lucas` en la sección 9.

Es la misma historia que contó *keyness* en la sección 7, y la misma que explicaba los
cuatro años positivos del sentimiento. **Tres métodos distintos, un solo hallazgo.**

> **Ojo con la convergencia.** Que tres métodos coincidan no los vuelve verdaderos: los
> tres miran la misma cosa, que es un cambio de vocabulario. Ninguno de los tres sabe que
> Casciari fundó una editorial. Eso lo sabés vos.

Y ahora, lo que ningún tutorial de tópicos hace: **ir a leer**.

In [ ]:
for numero_de_topico in range(CUANTOS_TOPICOS):
    pesos_del_topico = reparto_por_parrafo[:, numero_de_topico]
    parrafo_mas_fuerte = int(np.argmax(pesos_del_topico))
    texto = corpus["texto"].iloc[parrafo_mas_fuerte]
    anio = corpus["anio"].iloc[parrafo_mas_fuerte]
    print("tópico", numero_de_topico, "|", anio)
    print("   ", texto[:180])
    print()

El párrafo más representativo del tópico 1 es *«LUCAS.- No, Alex. No es eso.»*, **y es de
2014**: un parlamento de obra de teatro, justo en el año en que el teatro vuelve. El modelo
agarró bien la forma dialogada.

El del tópico 2 es la tapa del primer número de Orsai, con los 10.080 lectores.

**Leer los casos confirma dos tópicos y desconfirma uno.** Sin esta celda, la tabla de
cuatro filas se habría reportado como si los cuatro valieran lo mismo.

> ### Perilla
>
> `CUANTOS_TOPICOS` es la perilla más cara del cuaderno, porque **el modelo nunca se queja**.
> Le pedís tres y da tres; le pedís diez y da diez.
>
> Corré la celda y fijate qué le pasa a las obras de teatro y a Orsai en cada caso.

In [ ]:
for cuantos in [2, 3, 6]:
    modelo_de_prueba = NMF(n_components=cuantos, random_state=SEMILLA,
                           init="nndsvd", max_iter=400)
    modelo_de_prueba.fit(pesos_por_parrafo)
    print("--- k =", cuantos)
    numero_de_topico = 0
    for componente in modelo_de_prueba.components_:
        palabras = palabras_del_topico(componente, vocabulario, cuantas=6)
        print("   ", numero_de_topico, ":", ", ".join(palabras))
        numero_de_topico = numero_de_topico + 1
    print()

Con `k=2` sobrevive **el teatro**, y la revista se disuelve dentro del tópico genérico: el
modelo se queda con el tema más distinto, no con el más importante.

Con `k=6` los dos temas reales siguen intactos y los **cuatro** tópicos restantes son
variantes del mismo relleno: `si, saber, dice, ver`, `había, dijo, chiri, casa`,
`siempre, nunca, día, cosas`. Le pedimos seis y nos dio seis.

No hay un `k` correcto que el algoritmo pueda encontrar por vos. Hay un `k` que responde tu
pregunta, y se justifica leyendo los casos.

### ✎ Para pensar

- El tópico 3 (`si`, `siempre`, `ser`, `mundo`) está hecho de palabras que quedaron sueltas.
  ¿Qué le pasaría si agregaras al vectorizador las 22 candidatas de la sección 6?
- ¿Qué pregunta de investigación **no** podés responder con tópicos sobre párrafos, y sí
  podrías si el corpus hubiera conservado los textos individuales?

## 13. De cuaderno a módulo

Corriste el proceso entero. Todas las funciones que definiste están en las celdas de arriba
y **se van a perder cuando cierres el kernel**.

Al lado de este cuaderno hay un archivo, `utils.py`, con esas mismas funciones. No es una
copia de respaldo: es la forma en que el código deja de ser un cuaderno y se vuelve algo
reutilizable.

**Por qué importa.** Un cuaderno sirve para explorar y mostrar. Cuando una función ya está
entendida y estable, se muda a un `.py`, y desde ahí la usan otros cuadernos, un script
programado y el trabajo del año que viene. Esto es lo que hace la diferencia entre *saber
Python* y *trabajar con Python*.

Leelo entero: **de arriba a abajo, es el proceso.**

In [ ]:
archivo = open("utils.py", encoding="utf-8")
contenido = archivo.read()
archivo.close()

for linea in contenido.split("\n"):
    if linea.startswith("# \u2550") or linea.startswith("def "):
        print(linea)

Seis etapas, trece funciones, un archivo. Y ahora la pregunta honesta: **¿son las mismas
funciones que definimos acá, o se desincronizaron?**

No hay que confiar: hay que verificar. Es para esto que existen las aserciones.

In [ ]:
import utils

# Comparamos las funciones del cuaderno contra las del módulo, sobre datos reales.
ejemplo = corpus["texto"].iloc[100]

assert utils.limpiar(ejemplo) == limpiar(ejemplo), "limpiar() no coincide con utils.limpiar()"

del_modulo = utils.frecuencias_de(palabras_por_anio, "padre")
del_cuaderno = frecuencias_de(palabras_por_anio, "padre")
assert del_modulo.equals(del_cuaderno), "frecuencias_de() no coincide con la del módulo"

del_modulo = utils.concordancia_de(corpus, "che", ventana=45)
assert len(del_modulo) == len(concordancia_che), "concordancia_de() no coincide"

del_modulo = utils.puntuar_texto(limpiar(ejemplo), polaridad_de_palabra)
del_cuaderno = puntuar_texto(limpiar(ejemplo), polaridad_de_palabra)
assert del_modulo == del_cuaderno, "puntuar_texto() no coincide con la del módulo"

print("Las cuatro funciones del módulo dan exactamente lo mismo que las del cuaderno.")

Eso es una **prueba de equivalencia**, y es la única razón por la que se puede tener el
mismo código en dos lugares sin que se convierta en un problema. Si alguien toca una de las
dos, esta celda falla.

El último paso es salir de Jupyter. Un script no necesita un cuaderno abierto: se puede
correr desde la terminal, y mañana lo puede correr una tarea programada a las ocho de la
mañana sin que haya nadie.

In [ ]:
%%writefile analisis_del_corpus.py
# ─────────────────────────────────────────────────────────────────────
# analisis_del_corpus.py
# Recorre el corpus y muestra las palabras características de un año.
# Uso:  python analisis_del_corpus.py
# ─────────────────────────────────────────────────────────────────────
import utils

ANIO = 2010


def main():
    corpus = utils.armar_corpus(list(range(2004, 2016)))
    corpus["texto_limpio"] = corpus["texto"].apply(utils.limpiar)

    palabras_vacias = utils.cargar_palabras_vacias()
    palabras_por_anio = utils.agrupar_por_anio(corpus)
    contenido = utils.conteos_de_contenido(palabras_por_anio, palabras_vacias)

    print("párrafos:", len(corpus))
    print("palabras características de", ANIO)
    print(utils.caracteristicas_de(contenido["conteos"], contenido["totales"],
                                   ANIO, cuantas=6))


if __name__ == "__main__":
    main()

In [ ]:
# 'sys.executable' es la ruta del Python que está corriendo este cuaderno, o sea el del
# entorno virtual. En una terminal con el entorno activo esto es, simplemente:
#     python analisis_del_corpus.py
import sys

!"{sys.executable}" analisis_del_corpus.py

### ✎ Para pensar

- ¿Qué hace `if __name__ == "__main__"`? ¿Qué pasaría si el script no lo tuviera y otro
  cuaderno hiciera `import analisis_del_corpus`?
- El script ya corre sin Jupyter. ¿Qué faltaría para que corra **todos los lunes a las ocho
  de la mañana** sin que nadie lo ejecute?
- El script usa `.apply(utils.limpiar)`, con el nombre de una función. ¿Por qué eso se
  puede leer sin esfuerzo y `.apply(lambda t: utils.limpiar(t))` no?

## 14. Ejercicio

Elegí **un término** que te parezca que dice algo sobre este corpus. Puede ser una palabra,
un bigrama o un trigrama.

El orden de los pasos importa, y el paso 2 va **antes** del 3: si mirás la concordancia
primero, ya no podés saber qué habrías concluido sin ella.

In [ ]:
# ─── Completá los pasos faltantes ──────────────────────────────────────────
#
# TODO 1: guardá tu término en 'mi_termino'.
# TODO 2: contá cuántas veces aparece por año con frecuencias_de() y mostralo.
# TODO 3: escribí en un comentario qué creés que significa ese número.
# TODO 4: recién ahí corré concordancia_de() y leé las apariciones.
# TODO 5: escribí si lo que leíste sostiene lo que habías escrito, o no.
#
# ───────────────────────────────────────────────────────────────────────────

mi_termino = ""

## 15. Espacio de práctica

In [ ]:
# ─── Espacio de práctica ───────────────────────────────────────────────────
#
# Consigna: el lexicón afectivo tiene 291 palabras y la sección 11 mostró que
# 'bueno' no debería estar. Mejoralo.
#
#   1. Elegí tres palabras del lexicón que te parezcan sospechosas en este
#      corpus, y auditá cada una con concordancia_de().
#   2. Editá 'datos/lexicon_afectivo_es.csv': cambiá los puntajes que
#      correspondan y agregá al menos cinco palabras rioplatenses que falten.
#   3. Volvé a correr la sección 11 completa con tu lexicón.
#   4. Escribí en markdown qué cambió, qué no cambió, y qué conclusión de la
#      sección 11 ya no se sostiene.
#
# ───────────────────────────────────────────────────────────────────────────

## Lo que hiciste hoy

- Corriste el proceso entero, de los doce archivos crudos hasta la comparación entre años,
  en un solo cuaderno.
- Encontraste un dato cableado —`posts_por_anio`— que no se puede verificar contra la
  fuente, y lo reemplazaste por uno que sí.
- Mediste que la **diversidad léxica** baja cuando el texto crece, y que por eso no compara
  años de distinto tamaño.
- Viste que un criterio automático de palabras vacías propone descartar `chiri`, que es una
  persona.
- Aprendiste la **concordancia**, la operación con la que empezaron las humanidades
  computacionales en 1949, y encontraste con ella dos cosas que ningún conteo mostraba:
  seis de los veinte *che* son Guevara, y hay doce párrafos duplicados.
- Hiciste un **análisis de sentimiento** con un lexicón abierto, mediste su cobertura real
  (1,5 %) y lo rompiste sacándole una sola palabra.
- Modelaste **tópicos** con NMF, les pusiste nombre vos, y comprobaste leyendo los casos
  que dos de los cuatro eran temas de verdad.
- Verificaste que `utils.py` hace exactamente lo mismo que el cuaderno, y sacaste el
  proceso de Jupyter con un script.

## Las limitaciones, en una tabla

| Método | Qué no puede ver | Dónde se arregla |
|---|---|---|
| Bag of words | El orden de las palabras y la sintaxis | Unidad 4 |
| Conteo crudo | Que los años tienen tamaños distintos | Normalizar por 10k |
| Diversidad léxica | Que depende del largo del texto | Recortar al mismo largo, o MTLD |
| Lexicón afectivo | Ironía, negación lejana, palabras que no están en la lista | Modelos entrenados |
| Lexicón sin lematizar | Que *quería* y *querer* son la misma palabra | Unidad 4 |
| Tópicos | Que un tópico puede no ser un tema | Leer los casos |
| Todo el cuaderno | Que *che* puede ser una persona | La concordancia |

## Lo que viene

En la unidad de spaCy el texto deja de ser una fila de caracteres y pasa a tener capas:
categoría gramatical, lema, morfología, sintaxis, entidades nombradas.

Ahí vas a poder distinguir el *che* interjección del *Che* nombre propio **sin leer las
veinte filas**, y el lexicón afectivo va a funcionar sobre lemas en vez de sobre las
variantes que alguien se acordó de escribir.

Pero fijate lo que no cambia: **el lugar donde decidís si el número dice lo que parece
decir sigue siendo el texto.** La herramienta se hace más fina; la obligación de ir a
mirar, no se va nunca.